## Group 1 - Scenarial 3: Health Triage Assistant
- Group Members
  - Allen Lyimo
  - Claverfred Mhidze
  - Frank Mtimbili

### Step 1: Set-up and Importing

In [2]:
#!pip install torch transformers datasets accelerate -q

import math
import re
import json
import torch
import shutil
import pandas as pd
from google.colab import files
from datasets import load_dataset
from datasets import Dataset
from transformers import GPT2LMHeadModel
from transformers import GPT2Tokenizer
from transformers import TrainingArguments, Trainer

### Step 2: Loading the Dataset

In [3]:
dataset = load_dataset("gretelai/symptom_to_diagnosis")
print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/2.46k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/171k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/42.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/853 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/212 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output_text', 'input_text'],
        num_rows: 853
    })
    test: Dataset({
        features: ['output_text', 'input_text'],
        num_rows: 212
    })
})
{'output_text': 'cervical spondylosis', 'input_text': "I've been having a lot of pain in my neck and back. I've also been having trouble with my balance and coordination. I've been coughing a lot and my limbs feel weak."}


### Step 2a: Checking Label List

In [4]:
# Cell: check label list
labels = train_df["output_text"].unique() if "train_df" in dir() else None
# if you haven't made train_df yet, use this instead:
labels = sorted(set(dataset["train"]["output_text"]))
print(len(labels))
for l in labels:
    print(l)

22
allergy
arthritis
bronchial asthma
cervical spondylosis
chicken pox
common cold
dengue
diabetes
drug reaction
fungal infection
gastroesophageal reflux disease
hypertension
impetigo
jaundice
malaria
migraine
peptic ulcer disease
pneumonia
psoriasis
typhoid
urinary tract infection
varicose veins


### Step 3: Cleaning the Dataset

In [5]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# basic cleaning
for df in [train_df, test_df]:
    df.drop_duplicates(subset="input_text", inplace=True)
    df.dropna(inplace=True)
    df["input_text"] = df["input_text"].str.strip()
    df["output_text"] = df["output_text"].str.strip().str.lower()

print(train_df.shape, test_df.shape)
train_df.head()

(849, 2) (212, 2)


,output_text,input_text
0,cervical spondylosis,I've been having a lot of pain in my neck and ...
1,impetigo,I have a rash on my face that is getting worse...
2,urinary tract infection,I have been urinating blood. I sometimes feel ...
3,arthritis,I have been having trouble with my muscles and...
4,dengue,I have been feeling really sick. My body hurts...


### Step 3a: Checking what got removed

In [6]:
original = pd.DataFrame(dataset["train"])
dupes = original[original.duplicated(subset="input_text", keep=False)]
print(dupes.sort_values("input_text")[["input_text", "output_text"]])

                                            input_text           output_text
233  I have been having back pain, a cough that won...  cervical spondylosis
698  I have been having back pain, a cough that won...  cervical spondylosis
318  I've been feeling really sick and tired. I've ...              jaundice
696  I've been feeling really sick and tired. I've ...              jaundice
368  I've been feeling really sick and tired. I've ...              jaundice
832  I've been feeling really sick and tired. I've ...              jaundice
17   I've been having a hard time breathing lately....      bronchial asthma
682  I've been having a hard time breathing lately....      bronchial asthma


### Step 4: Severity/Urgency Map

In [7]:
severity_map = {
    "allergy": "routine",
    "arthritis": "routine",
    "bronchial asthma": "urgent",
    "cervical spondylosis": "routine",
    "chicken pox": "routine",
    "common cold": "routine",
    "dengue": "urgent",
    "diabetes": "urgent",
    "drug reaction": "urgent",
    "fungal infection": "routine",
    "gastroesophageal reflux disease": "routine",
    "hypertension": "urgent",
    "impetigo": "routine",
    "jaundice": "urgent",
    "malaria": "urgent",
    "migraine": "routine",
    "peptic ulcer disease": "urgent",
    "pneumonia": "urgent",
    "psoriasis": "routine",
    "typhoid": "urgent",
    "urinary tract infection": "routine",
    "varicose veins": "routine",
}

# sanity check: confirm every label in the cleaned data has a mapping
labels = sorted(train_df["output_text"].unique())
missing = [l for l in labels if l not in severity_map]
print("Missing from severity_map:", missing)  # should be []
print("Total mapped:", len(severity_map))

Missing from severity_map: []
Total mapped: 22


### Step 5: Building Prompt and Response Pairs *(Teaching model how to talk)*

In [8]:
def format_example(row):
    diagnosis = row["output_text"]
    urgency = severity_map.get(diagnosis, "urgent")
    prompt = f"Patient symptoms: {row['input_text']}\nTriage assessment:"
    response = (f"\nBased on the symptoms described, this could possibly indicate "
                f"**{diagnosis}**.\nSuggested urgency: **{urgency}**.\n"
                f"This is a preliminary suggestion only — please have a trained professional confirm.")
    return prompt + response

train_df["text"] = train_df.apply(format_example, axis=1)
test_df["text"] = test_df.apply(format_example, axis=1)

print(train_df["text"].iloc[0])
print("---")
print(train_df["text"].iloc[5])

Patient symptoms: I've been having a lot of pain in my neck and back. I've also been having trouble with my balance and coordination. I've been coughing a lot and my limbs feel weak.
Triage assessment:
Based on the symptoms described, this could possibly indicate **cervical spondylosis**.
Suggested urgency: **routine**.
This is a preliminary suggestion only — please have a trained professional confirm.
---
Patient symptoms: I've been feeling really run down and weak. My throat is sore and I've been coughing a lot. I've also been having chills and a fever.
Triage assessment:
Based on the symptoms described, this could possibly indicate **common cold**.
Suggested urgency: **routine**.
This is a preliminary suggestion only — please have a trained professional confirm.


### Step 6: Tokenize

In [9]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default, so we reuse the "end of text" token (<END>)

train_ds = Dataset.from_pandas(train_df[["text"]])
test_ds = Dataset.from_pandas(test_df[["text"]])

def tokenize_fn(examples):
    out = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    out["labels"] = out["input_ids"].copy()
    return out

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
test_tokenized = test_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

print(train_tokenized[0])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/849 [00:00<?, ? examples/s]

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

{'__index_level_0__': 0, 'input_ids': [12130, 1153, 7460, 25, 314, 1053, 587, 1719, 257, 1256, 286, 2356, 287, 616, 7393, 290, 736, 13, 314, 1053, 635, 587, 1719, 5876, 351, 616, 5236, 290, 19877, 13, 314, 1053, 587, 48308, 257, 1256, 290, 616, 21755, 1254, 4939, 13, 198, 51, 4087, 8922, 25, 198, 15001, 319, 262, 7460, 3417, 11, 428, 714, 5457, 7603, 12429, 66, 712, 605, 599, 623, 2645, 5958, 1174, 13, 198, 43857, 276, 25615, 25, 12429, 81, 28399, 1174, 13, 198, 1212, 318, 257, 15223, 13052, 691, 851, 3387, 423, 257, 8776, 4708, 6216, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

### Step 7: Loading Pretrained GPT-2

In [10]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Embedding(50257, 768)

### Step 8: Fine-tune

In [11]:
training_args = TrainingArguments(
    output_dir="./gpt2-triage",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    eval_strategy="epoch",
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.560148,0.483039
2,0.444505,0.440736
3,0.400764,0.422076
4,0.364043,0.416502
5,0.343104,0.414616


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1065, training_loss=0.4468436805295273, metrics={'train_runtime': 314.0582, 'train_samples_per_second': 13.517, 'train_steps_per_second': 3.391, 'total_flos': 277296168960000.0, 'train_loss': 0.4468436805295273, 'epoch': 5.0})

### Step 8a: Confirming the best epoch's weight loaded

In [12]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

./gpt2-triage/checkpoint-1065
0.4146156907081604


### Step 9: Model Evaluation *(Checking how confused the model was, on average)*

In [13]:
eval_results = trainer.evaluate()
print(eval_results)
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
0.343104,0.414616,5


{'eval_loss': 0.4146156907081604}
Perplexity: 1.51


### Step 10: Model Testing and Listing Emergency Keywords

### Step 10a: Define Emergency Keywords List

In [14]:
EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "difficulty breathing",
    "severe bleeding", "unconscious", "unresponsive", "stroke", "seizure",
    "suicidal", "suicide", "overdose", "severe allergic reaction", "anaphylaxis",
    "blue lips", "crushing pain", "not breathing", "collapsed",
    "worst headache of my life", "sudden confusion", "can't move"
]

SELF_HARM_KEYWORDS = [
    "hurt myself", "kill myself", "suicidal", "suicide", "end my life",
    "don't want to be here", "don't want to live", "want to die",
    "self harm", "self-harm", "harm myself"
]

### Step 10b: Symptom list setup

In [15]:
SYMPTOM_WHITELIST = {
    "head", "neck", "back", "chest", "stomach", "abdomen", "throat", "skin",
    "eye", "eyes", "ear", "ears", "nose", "mouth", "joint", "joints", "knee",
    "knees", "leg", "legs", "arm", "arms", "hand", "hands", "foot", "feet",
    "muscle", "muscles", "limb", "limbs", "lung", "lungs", "heart",
    "liver", "kidney", "bladder", "spine", "shoulder", "hip", "ankle", "wrist",
    "pain", "ache", "aches", "aching", "achy", "sore", "soreness", "hurt", "hurting",
    "fever", "chills", "sweating", "sweaty", "nausea", "nauseous", "nauseated",
    "vomiting", "vomit", "queasy", "clammy",
    "dizzy", "dizziness", "lightheaded", "faint", "fainting", "woozy",
    "fatigue", "tired", "tiredness", "exhausted", "exhaustion",
    "weak", "weakness", "drained", "lethargic", "lethargy",
    "cough", "coughing", "sneeze", "sneezing", "sniffly", "runny", "stuffy",
    "hoarse", "wheeze", "wheezing",
    "rash", "itch", "itchy", "itching",
    "swelling", "swollen", "puffy", "bloating", "bloated",
    "cramp", "cramps", "cramping", "spasm", "spasms",
    "stiff", "stiffness", "burning", "stinging", "throbbing", "pounding",
    "numb", "numbness", "tingling", "pins",
    "breathless", "breathing", "congestion", "congested",
    "diarrhea", "constipation", "constipated",
    "bleeding", "bruise", "bruising", "blister", "blisters",
    "headache", "migraine", "cold", "flu", "infection", "inflammation",
    "discharge", "discomfort", "irritation", "irritated",
    "spots", "patches", "scaly", "flaky", "dry", "peeling",
    "red", "redness", "yellow", "yellowish", "jaundice", "pale", "pallor",
    "thirst", "thirsty", "urination", "urine", "urinate", "urinating",
    "appetite", "insomnia", "sleepless", "sleepy", "drowsy",
    "anxious", "restless", "confusion", "confused", "disoriented",
    "blurry", "blurred", "vision", "hearing", "balance", "coordination",
    "seizure", "tremor", "shaking", "trembling",
    "palpitations", "pressure", "tightness", "tight",
    "cold", "hot", "sweats", "shivering", "shivers",
    "runny", "blocked", "sinus", "phlegm", "mucus",
    "bump", "lump", "sting", "cut", "wound", "bloody",
}

def is_plausible_symptom_text(text, min_overlap=1):
    words = set(re.findall(r"[a-z]+", text.lower()))
    overlap = words & SYMPTOM_WHITELIST
    return len(overlap) >= min_overlap, overlap

### Step 10c: Triage Function, Combining the Safety Check and confidence threshold

In [16]:
CONFIDENCE_THRESHOLD = 0.95

# Main triage function
def triage(symptom_text):
    text_lower = symptom_text.lower()

    # 1. Self-harm check — highest priority, never touches the model
    for kw in SELF_HARM_KEYWORDS:
        if kw in text_lower:
            return ("It sounds like you may be going through something very difficult right now.\n"
                    "Please reach out for support:\n"
                    "• Call or text 112 (Suicide & Crisis Lifeline)\n"
                    "• Or contact your local emergency number\n"
                    "This is a rule-based safety response — it did not come from the model.")

    # 2. Physical emergency check — bypasses the model
    for kw in EMERGENCY_KEYWORDS:
        if kw in text_lower:
            return (f"🚨 EMERGENCY DETECTED (matched: '{kw}').\n"
                    f"Please seek immediate emergency care or call emergency services now.\n"
                    f"This is a rule-based safety override — it did not come from the model.")

    # 3. Plausibility check — is this even symptom-like text?
    plausible, overlap = is_plausible_symptom_text(symptom_text)
    if not plausible:
        return (f"⚠️ This doesn't appear to describe medical symptoms clearly enough to assess.\n"
                f"Please describe what you're physically experiencing, "
                f"or consult a trained professional directly.")

    # 4. Model generation (only reached if text passed all checks above)
    prompt = f"Patient symptoms: {symptom_text}\nTriage assessment:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs, max_new_tokens=60, do_sample=True, top_p=0.9,
        temperature=0.4, pad_token_id=tokenizer.eos_token_id,
        output_scores=True, return_dict_in_generate=True
    )

    scores = output.scores
    generated_ids = output.sequences[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(generated_ids, skip_special_tokens=True)

    cutoff_text = result.split("This is a preliminary")[0]
    cutoff_token_count = len(tokenizer(cutoff_text)["input_ids"])

    probs = []
    for i in range(min(cutoff_token_count, len(scores))):
        prob_dist = torch.softmax(scores[i][0], dim=-1)
        token_id = generated_ids[i]
        probs.append(prob_dist[token_id].item())

    avg_confidence = sum(probs) / len(probs) if probs else 0

    if avg_confidence < CONFIDENCE_THRESHOLD:
        return (f"⚠️ The system could not confidently assess these symptoms "
                f"(confidence: {avg_confidence:.2f}).\n"
                f"This description doesn't clearly match a known pattern — "
                f"please consult a trained professional directly rather than relying on this suggestion.\n\n"
                f"[Low-confidence model output, shown for reference only: {result}]")

    return f"{result}\n\n(Model confidence: {avg_confidence:.2f})"

### Step 10d: Running Test Cases *(Both Normal and Emergency case)*

In [17]:
print("=== Test 1: Mild symptoms ===")
print(triage("I have a mild headache and a runny nose"))

print("\n=== Test 2: Emergency symptoms ===")
print(triage("I'm having severe chest pain and can't breathe"))

=== Test 1: Mild symptoms ===

Based on the symptoms described, this could possibly indicate **migraine**.
Suggested urgency: **routine**.
This is a preliminary suggestion only — please have a trained professional confirm.

(Model confidence: 1.00)

=== Test 2: Emergency symptoms ===
🚨 EMERGENCY DETECTED (matched: 'chest pain').
Please seek immediate emergency care or call emergency services now.
This is a rule-based safety override — it did not come from the model.


### Step 10e: Performing few more test cases

In [18]:
tests = [
    "I have joint pain and stiffness in my knees",
    "I've had a burning sensation when I urinate and lower belly pain",
    "I have an itchy red rash with scaly patches on my skin",
    "I think I might hurt myself, I don't want to be here anymore",
]

for t in tests:
    print(f"--- Input: {t} ---")
    print(triage(t))
    print()

--- Input: I have joint pain and stiffness in my knees ---

Based on the symptoms described, this could possibly indicate **arthritis**.
Suggested urgency: **routine**.
This is a preliminary suggestion only — please have a trained professional confirm.

(Model confidence: 0.99)

--- Input: I've had a burning sensation when I urinate and lower belly pain ---

Based on the symptoms described, this could possibly indicate **urinary tract infection**.
Suggested urgency: **routine**.
This is a preliminary suggestion only — please have a trained professional confirm.

(Model confidence: 1.00)

--- Input: I have an itchy red rash with scaly patches on my skin ---

Based on the symptoms described, this could possibly indicate **impetigo**.
Suggested urgency: **routine**.
This is a preliminary suggestion only — please have a trained professional confirm.

(Model confidence: 1.00)

--- Input: I think I might hurt myself, I don't want to be here anymore ---
It sounds like you may be going through

### Step 11: Deploying the Model

### Step 11a: Saving the fine-tuned model and tokenizer

In [19]:
model.save_pretrained("./gpt2-triage-final")
tokenizer.save_pretrained("./gpt2-triage-final")
print("Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


### Step 11b: Zip and download the model

In [20]:
shutil.make_archive("gpt2-triage-final", "zip", "gpt2-triage-final") # Zip the saved model

files.download("gpt2-triage-final.zip") # Download the model

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>